In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Load model comparison
comparison = pd.read_csv('../data/model_comparison.csv', index_col=0)
print("Model Comparison")
print(comparison)

# Load the best model (XGBoost based on results)
model_data = joblib.load('../models/xgboost_model.pkl')

model = model_data['model']
scaler = model_data['scaler']
feature_names = model_data['feature_names']

print("Model loaded duccessfully!")
print(f"Feature names: {feature_names}")

# Create sample features for a match
# Using Manchester City (home) vs Arsenal (away)
sample_features = {
    'home_form_5': 12.0,  # 4 wins, 0 draws = 12 points
    'away_form_5': 9.0,   # 3 wins, 0 draws = 9 points
    'home_goals_scored_avg': 2.5,
    'home_goals_conceded_avg': 0.8,
    'away_goals_scored_avg': 2.0,
    'away_goals_conceded_avg': 1.0,
    'h2h_home_wins': 2,
    'h2h_draws': 1,
    'h2h_away_wins': 2,
    'home_win_rate': 0.75,
    'away_win_rate': 0.65,
    'days_since_home_last_match': 7,
    'days_since_away_last_match': 7
}

import numpy as np

# Prepare features
X_sample = pd.DataFrame([sample_features])[feature_names]
X_scaled = scaler.transform(X_sample)

# Predict
prediction = model.predict(X_scaled)[0]
probabilities = model.predict_proba(X_scaled)[0]

outcome_map = {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}

print("\nSample Prediction:")
print(f"Match: Strong Home Team vs Strong Away Team")
print(f"\nPredicted Outcome: {outcome_map[prediction]}")
print(f"\nProbabilities:")
print(f"  Home Win: {probabilities[2]:.1%}")
print(f"  Draw: {probabilities[1]:.1%}")
print(f"  Away Win: {probabilities[0]:.1%}")
print(f"\nConfidence: {max(probabilities):.1%}")

# Calculate what "random guessing" would achieve
# If we always predict "Home Win" (most common outcome)
df_features = pd.read_csv('../data/processed/match_features.csv')
baseline_accuracy = (df_features['target'] == 'HOME_TEAM').mean()

print(f"\nBaseline (always predict home win): {baseline_accuracy:.1%}")
print(f"Our model accuracy: {comparison.loc['xgboost', 'accuracy']:.1%}")
print(f"Improvement over baseline: {(comparison.loc['xgboost', 'accuracy'] - baseline_accuracy):.1%}")

insights = f"""
MODEL TRAINING INSIGHTS:

1. Best Model: XGBoost with {comparison.loc['xgboost', 'accuracy']:.1%} accuracy

2. Performance:
   - Accuracy: {comparison.loc['xgboost', 'accuracy']:.1%}
   - F1 Score: {comparison.loc['xgboost', 'f1_weighted']:.1%}
   - Better than random (33.3%): {((comparison.loc['xgboost', 'accuracy'] - 0.333) / 0.333 * 100):.0f}%
   - vs Baseline (home win): {((comparison.loc['xgboost', 'accuracy'] - baseline_accuracy) * 100):.1f} percentage points

3. What This Means:
   - Model predicts better than random chance (33.3%)
   - Currently BELOW the home-win baseline ({baseline_accuracy:.1%})
   - Expected with only {len(df_features)} training samples
   - Football prediction needs 2000+ matches for reliable results
   - Professional betting models achieve 55-65% with massive datasets

4. Most Important Features:
   - Team form (recent match results)
   - Goals scored/conceded averages
   - Home/away win rates
   - Head-to-head history
   - Rest days between matches

5. Why Below Baseline:
   - Only {len(df_features)} matches — too few for ML to learn robust patterns
   - Model spreads probability across 3 classes instead of
     always picking the most common one (home win)
   - With more data, model will surpass the naive baseline

6. Next Steps:
   - Add historical data (football-data.co.uk) for 2000+ matches
   - Add standings position as a feature
   - Run hyperparameter tuning (GridSearchCV)
   - Deploy model via Flask API
   - Track predictions vs actual results over time
"""

print(insights)

with open('../data/model_anylysis.txt', 'w') as f:
    f.write(insights)

Model Comparison
                     accuracy  f1_weighted  precision_weighted  \
xgboost              0.389831     0.383640            0.386762   
gradient_boosting    0.355932     0.343777            0.337072   
random_forest        0.338983     0.284050            0.271738   
logistic_regression  0.338983     0.308165            0.301872   

                     recall_weighted  
xgboost                     0.389831  
gradient_boosting           0.355932  
random_forest               0.338983  
logistic_regression         0.338983  
Model loaded duccessfully!
Feature names: ['home_form_5', 'away_form_5', 'home_goals_scored_avg', 'home_goals_conceded_avg', 'away_goals_scored_avg', 'away_goals_conceded_avg', 'h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'home_win_rate', 'away_win_rate', 'days_since_home_last_match', 'days_since_away_last_match']

Sample Prediction:
Match: Strong Home Team vs Strong Away Team

Predicted Outcome: Away Win

Probabilities:
  Home Win: 35.8%
  Draw: 28.2